In [2]:
import sys
print(sys.executable)

c:\Users\guilh\OneDrive\Área de Trabalho\Faculdade\6 Periodo\Mineração de Bitcoin\Trabalho_Final\.venv\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

## INGESTAO E TRATAMENTO

#### PASSO 1: FUNÇÃO EXPORTADORA PARA ARFF (WEKA)

In [5]:
def exportar_arff(df, nome_arquivo, nome_relacao):
    """
    Função em Python para converter um DataFrame Pandas 
    diretamente no formato exigido pelo Weka (.arff).
    """
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(f"@relation {nome_relacao}\n\n")
        
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]) and col != "CLASSE_DEFAULT":
                f.write(f"@attribute {col} numeric\n")
            else:
                valores_unicos = sorted(df[col].astype(str).unique().tolist())
                dominio = ",".join([f"'{v}'" if " " in v else v for v in valores_unicos])
                f.write(f"@attribute {col} {{{dominio}}}\n")
                
        f.write("\n@data\n")
        for _, row in df.iterrows():
            linha = ",".join([str(val) for val in row.values])
            f.write(f"{linha}\n")
    print(f" -> Arquivo salvo com sucesso: {nome_arquivo}")

#### PASSO 2: INGESTÃO E LIMPEZA (id 350 - Default of Credit Card Clients Dataset)

In [6]:
dataset = fetch_ucirepo(id=350)
df = dataset.data.features.copy()
df.columns = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", 
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6", 
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]
df["CLASSE_DEFAULT"] = dataset.data.targets.iloc[:, 0].astype(int)

# Removendo identificador 
if "ID" in df.columns:
    df.drop(columns=["ID"], inplace=True)

# Tratando códigos anômalos não documentados
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

for col in ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]:
    df[col] = df[col].replace({-2: 0, -1: 0}) # Junta quem não deve e quem pagou em dia

# Converte categóricos para String (o Weka entenderá como Nominal automaticamente)
df["SEX"] = df["SEX"].map({1: "Masculino", 2: "Feminino"})
df["EDUCATION"] = df["EDUCATION"].map({1: "PosGrad", 2: "Universidade", 3: "EnsinoMedio", 4: "Outros"})
df["MARRIAGE"] = df["MARRIAGE"].map({1: "Casado", 2: "Solteiro", 3: "Outros"})
for col in ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]:
    df[col] = df[col].apply(lambda x: "0_EmDia" if x == 0 else f"Atraso_{x}m")

DatasetNotFoundError: Error reading data csv file for "Default of Credit Card Clients" dataset (id=350).

#### PASSO 3: AMOSTRAGEM ESTRATIFICADA

In [ ]:
print("[2/4] Realizando Amostragem Estratificada (6.000 instâncias)...")
df_exec1, _ = train_test_split(
    df, train_size=6000, stratify=df["CLASSE_DEFAULT"], random_state=42
)
df_exec1 = df_exec1.reset_index(drop=True)

#### PASSO 4: DISCRETIZAÇÃO PARA A EXECUÇÃO 2

In [ ]:
df_exec2 = df_exec1.copy()

# Faixas para LIMIT_BAL e AGE
df_exec2["LIMIT_BAL"] = pd.cut(df_exec1["LIMIT_BAL"], bins=[-np.inf, 50000, 150000, 250000, np.inf], 
                               labels=["F1_Ate50k", "F2_50k_a_150k", "F3_150k_a_250k", "F4_Acima250k"]).astype(str)

df_exec2["AGE"] = pd.cut(df_exec1["AGE"], bins=[-np.inf, 29, 39, 49, np.inf], 
                         labels=["F1_Jovem", "F2_Adulto", "F3_Maduro", "F4_Senior"]).astype(str)

# Faixas financeiras
for i in range(1, 7):
    # BILL_AMT (Fatura)
    col_bill = f"BILL_AMT{i}"
    df_exec2[col_bill] = pd.cut(df_exec1[col_bill], bins=[-np.inf, 0, 20000, 70000, np.inf], 
                                labels=["F1_SemFatura", "F2_Baixa", "F3_Media", "F4_Alta"]).astype(str)
    # PAY_AMT (Valor Pago)
    col_pay = f"PAY_AMT{i}"
    df_exec2[col_pay] = pd.cut(df_exec1[col_pay], bins=[-np.inf, 0, 2000, 6000, np.inf], 
                               labels=["F1_ZeroPago", "F2_Minimo", "F3_Parcial", "F4_Alto"]).astype(str)

#### PASSO 5: EXPORTAÇÃO DOS ARQUIVOS .ARFF

In [ ]:
exportar_arff(df_exec1, "base22_exec1_original.arff", "Credit_Exec1")
exportar_arff(df_exec2, "base22_exec2_discretizada.arff", "Credit_Exec2")
